# Custom Image Mask Generation & Visualisation

This notebook loads the 4 specific target images, queries their actual ground truth masks from the COCO annotations on the `F:` drive (mounted under `/mnt/f`), runs model inference (UNet-Lite), displays the inputs, ground truth, and prediction side-by-side, and saves the mask predictions to the local directory.

In [ ]:
# --- imports & setup ---
import sys, os
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image as PILImage

# Ensure project root is in path
current = Path.cwd()
project_root = current
for parent in [current] + list(current.parents):
    if (parent / ".git").exists() or (parent / "README.md").exists():
        project_root = parent
        break
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.environ.setdefault("PYTHONPATH", str(project_root))
os.chdir(str(project_root))

from src.model import UNetLite
from src.config import Config
from tools.data import create_dataloaders, label_to_color, denormalize

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# --- configuration ---
cfg = Config()
# Target checkpoint
checkpoint_path = "models/unet_base_copy/best_unet_base.pth"

print(f"Dataset Path: {cfg.val_ann_file}")
print(f"Model Checkpoint: {checkpoint_path}")

In [ ]:
# --- load dataloader ---
print("Loading annotations and initializing dataloader...")
_, val_loader, info = create_dataloaders(
    "coco_person",
    image_size=cfg.image_size,
    batch_size=1,
    num_workers=0,
)

In [ ]:
# --- load model ---
print("Building and loading model...")
model = UNetLite(None, use_attention=False).to(device)
if os.path.exists(checkpoint_path):
    ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    state_dict = ckpt.get("model_state_dict", ckpt)
    model.load_state_dict(state_dict, strict=False)
    print("Checkpoint loaded successfully!")
else:
    print("WARNING: Checkpoint not found, using random weights!")
model.eval()

In [ ]:
# --- run inference and plot results ---
target_indices = [110, 884, 1316, 1798]
img_names = ["000000008532", "000000002685", "000000004134", "000000005529"]

fig, axes = plt.subplots(4, 3, figsize=(15, 18))

for idx, val_idx in enumerate(target_indices):
    # Check if we should use local fallback
    use_local_fallback = False
    if val_idx >= len(val_loader.dataset):
        use_local_fallback = True
    else:
        try:
            img_tensor, mask_tensor = val_loader.dataset[val_idx]
        except Exception as e:
            use_local_fallback = True
            
    if use_local_fallback:
        from PIL import Image as PILImage
        import torchvision.transforms.functional as TF
        from src.dataset import IMAGENET_MEAN, IMAGENET_STD
        from torchvision.transforms import InterpolationMode
        
        local_img_path = f"/home/lenovo/a3_dl_fn/UNet_Training_From_Scratch/{img_names[idx]}.jpg"
        image = PILImage.open(local_img_path).convert("RGB")
        short_edge = min(cfg.image_size)
        img = TF.resize(image, short_edge, interpolation=InterpolationMode.BILINEAR)
        img_tensor = TF.center_crop(img, cfg.image_size)
        img_tensor = TF.to_tensor(img_tensor)
        img_tensor = TF.normalize(img_tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD)
        mask_tensor = torch.zeros(1, *cfg.image_size)
        
    # Prepare inputs
    img_gpu = img_tensor.unsqueeze(0).to(device)
    
    # Run prediction
    with torch.no_grad():
        logits = model(img_gpu)
        if logits.shape[1] == 1:
            pred = (logits > 0).cpu().int()
        else:
            pred = logits.argmax(dim=1).cpu()
            
    img_np = denormalize(img_tensor)
    gt = mask_tensor.squeeze().numpy() if isinstance(mask_tensor, torch.Tensor) else mask_tensor
    pr = pred[0].squeeze().numpy()
    
    # Plot Input Image
    axes[idx, 0].imshow(img_np)
    axes[idx, 0].set_title(f"Sample {idx+1} Input ({img_names[idx]}.jpg)", fontweight="bold")
    axes[idx, 0].axis("off")
    
    # Plot Ground Truth
    axes[idx, 1].imshow(label_to_color(gt, info.palette))
    axes[idx, 1].set_title(f"Sample {idx+1} Ground Truth Mask", fontweight="bold")
    axes[idx, 1].axis("off")
    
    # Plot Prediction
    colored_pred = label_to_color(pr, info.palette)
    axes[idx, 2].imshow(colored_pred)
    axes[idx, 2].set_title(f"Sample {idx+1} Predicted Mask", fontweight="bold")
    axes[idx, 2].axis("off")
    
    # Save predictions to file next to the original files
    mask_save_path = f"{img_names[idx]}_mask.png"
    from PIL import Image as PILImage
    PILImage.fromarray(colored_pred).save(mask_save_path)
    print(f"Saved predicted mask to: {mask_save_path}")

# Add Legend
palette_01 = info.palette / 255.0
patches = [mpatches.Patch(color=palette_01[i], label=name)
           for i, name in enumerate(info.class_names)]
fig.legend(handles=patches, loc="lower center", ncol=4, frameon=False,
           bbox_to_anchor=(0.5, 0.05), fontsize=10)

plt.tight_layout()
os.makedirs("paper_figures", exist_ok=True)
fig.savefig("paper_figures/07_custom_predictions.pdf", bbox_inches="tight", dpi=150)
print("Saved dashboard visualization to: paper_figures/07_custom_predictions.pdf")
plt.show()